## Your First Model - Follow-Along Practice

This notebook covers the practice examples for Pydantic BaseModel, Dataclasses, Type Coercion, and Validation.

#### Install Dependencies

Run the following cell to install Pydantic in your notebook environment.

In [1]:
!pip install pydantic

#### 1. Creating a Basic Model

A Pydantic model is a class that defines the structure of your data. It specifies what fields exist and what types they should be.

In [2]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    email: str
    age: int

#### 2. Dataclasses vs Pydantic

Python has built-in dataclasses for data structures, but they do not validate anything. Pydantic models look similar but actually enforce the types.

In [3]:
from dataclasses import dataclass

@dataclass
class UserDataclass:
    name: str
    email: str
    age: int

user = UserDataclass(name="Alice", email="alice@example.com", age="not a number")
print(user.age)  # "not a number" - no validation!

not a number


In [4]:
from pydantic import BaseModel, ValidationError

class UserPydantic(BaseModel):
    name: str
    email: str
    age: int

try:
    user = UserPydantic(name="Alice", email="alice@example.com", age="not a number")
except ValidationError as e:
    print("ValidationError raised as expected:")
    print(e)

ValidationError raised as expected:
1 validation error for UserPydantic
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/int_parsing


#### 3. Creating Instances

Create a model instance by passing data to it.

In [ ]:
class User(BaseModel):
    name: str
    email: str
    age: int

# Create a user
user = User(name="Alice", email="alice@example.com", age=30)

print(user.name)   # Alice
print(user.email)  # alice@example.com
print(user.age)    # 30

#### 4. Validation in Action

Passing invalid data raises a validation error immediately.

In [ ]:
try:
    # This will raise a validation error
    user = User(name="Alice", email="alice@example.com", age="thirty")
except ValidationError as e:
    print(e)

#### 5. Automatic Type Coercion

Pydantic converts compatible types automatically.

In [ ]:
class UserCoercion(BaseModel):
    name: str
    age: int

# String "25" becomes integer 25
user = UserCoercion(name="Alice", age="25")
print(user.age)        # 25
print(type(user.age))  # <class 'int'>

#### 6. Required vs Optional Fields

Fields without defaults are required.

In [ ]:
class UserOptional(BaseModel):
    name: str              # Required
    email: str             # Required
    age: int | None = None # Optional (has default)

# Works - age is optional
user1 = UserOptional(name="Alice", email="alice@example.com")
print("user1.age:", user1.age)  # None

# Also works - providing age
user2 = UserOptional(name="Bob", email="bob@example.com", age=25)
print("user2.age:", user2.age)  # 25

#### 7. Default Values

Set defaults for fields that usually have a common value.

In [ ]:
class APIConfig(BaseModel):
    api_key: str
    model: str = "gpt-4"
    max_tokens: int = 1000
    temperature: float = 0.7

# Only api_key is required
config = APIConfig(api_key="sk-abc123")

print(config.model)       # gpt-4
print(config.max_tokens)  # 1000

#### 8. Converting to Dict & JSON

Use `model_dump()` for a dict and `model_dump_json()` for a JSON string.

In [ ]:
user = User(name="Alice", email="alice@example.com", age=30)

# Convert to dict
user_dict = user.model_dump()
print("Dict:", user_dict)

# Convert to JSON string
json_string = user.model_dump_json()
print("JSON:", json_string)

#### 9. Creating from a Dictionary

Create a model from dictionary using unpacking `**` or `model_validate()`.

In [ ]:
data = {"name": "Alice", "email": "alice@example.com", "age": 30}

# Option 1: Unpack the dict
user1 = User(**data)

# Option 2: Use model_validate
user2 = User.model_validate(data)

print(user1 == user2)  # True

#### 10. Models as Type Hints

Pydantic models work as type hints in your functions.

In [ ]:
class UserHint(BaseModel):
    name: str
    email: str

def greet_user(user: UserHint) -> str:
    return f"Hello, {user.name}!"

def load_user(data: dict) -> UserHint:
    return UserHint.model_validate(data)

user = load_user({"name": "Alice", "email": "alice@example.com"})
message = greet_user(user)
print(message)  # Hello, Alice!

#### 11. Real-World Example

A model for handling API responses.

In [ ]:
class WeatherResponse(BaseModel):
    city: str
    temperature: float
    humidity: int
    description: str

api_data = {
    "city": "Amsterdam",
    "temperature": 18.5,
    "humidity": 75,
    "description": "Partly cloudy"
}

weather = WeatherResponse.model_validate(api_data)
print(f"Weather in {weather.city}: {weather.temperature}°C")
print(f"Humidity: {weather.humidity}%")
print(f"Conditions: {weather.description}")

#### 12. Common Mistakes

#### Mutable Default Values

Pydantic automatically creates a new list for each instance, avoiding shared mutable state issues.

In [ ]:
class UserTags(BaseModel):
    tags: list[str] = []

user1 = UserTags()
user2 = UserTags()
user1.tags.append("python")
print(user2.tags)  # [] - safe and unaffected!

#### 13. Strict Mode

If you want to disable coercion and require exact types, use strict mode.

In [ ]:
from pydantic import ConfigDict

class StrictUser(BaseModel):
    model_config = ConfigDict(strict=True)
    
    name: str
    age: int

try:
    user = StrictUser(name="Alice", age="25")
except ValidationError as e:
    print("Strict mode prevented type coercion:")
    print(e)